In [5]:
# ============================================================
# PEARLS AQI PREDICTOR
# MODEL TRAINING PIPELINE
# STEP 1: IMPORTS AND CONFIGURATION
# ============================================================

import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# XGBoost
from xgboost import XGBRegressor

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("=" * 60)
print("PEARLS AQI PREDICTOR")
print("MODEL TRAINING PIPELINE")
print("=" * 60)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

PEARLS AQI PREDICTOR
MODEL TRAINING PIPELINE
TensorFlow version: 2.21.0
NumPy version: 2.4.2
Pandas version: 2.3.3


In [6]:
# ============================================================
# STEP 2: MODEL FEATURES
# ============================================================

feature_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",

    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_6",
    "aqi_lag_12",
    "aqi_lag_24",
    "aqi_lag_48",
    "aqi_lag_72",

    "pm2_5_lag_1",
    "pm2_5_lag_3",
    "pm2_5_lag_6",
    "pm2_5_lag_24",

    "pm10_lag_1",
    "pm10_lag_3",
    "pm10_lag_6",
    "pm10_lag_24",

    "carbon_monoxide_lag_1",
    "carbon_monoxide_lag_3",
    "carbon_monoxide_lag_6",
    "carbon_monoxide_lag_24",

    "nitrogen_dioxide_lag_1",
    "nitrogen_dioxide_lag_3",
    "nitrogen_dioxide_lag_6",
    "nitrogen_dioxide_lag_24",

    "sulphur_dioxide_lag_1",
    "sulphur_dioxide_lag_3",
    "sulphur_dioxide_lag_6",
    "sulphur_dioxide_lag_24",

    "ozone_lag_1",
    "ozone_lag_3",
    "ozone_lag_6",
    "ozone_lag_24",

    "aqi_3h_mean",
    "aqi_6h_mean",
    "aqi_12h_mean",
    "aqi_24h_mean",

    "pm2_5_3h_mean",
    "pm2_5_6h_mean",
    "pm2_5_24h_mean",

    "pm10_3h_mean",
    "pm10_6h_mean",
    "pm10_24h_mean",

    "carbon_monoxide_24h_mean",
    "nitrogen_dioxide_24h_mean",
    "sulphur_dioxide_24h_mean",
    "ozone_24h_mean",

    "aqi_change_1h",
    "aqi_change_3h",
    "aqi_change_6h",
    "aqi_change_24h",

    "pm2_5_change_1h",
    "pm2_5_change_24h",

    "pm10_change_1h",
    "pm10_change_24h"
]

TARGET = "target_aqi"

print("Number of model features:", len(feature_columns))
print("Target:", TARGET)

Number of model features: 70
Target: target_aqi


In [7]:
# ============================================================
# STEP 3: LOAD PROCESSED FEATURE DATASET
# ============================================================

DATA_PATH = r"D:\Internship\pearls-aqi-predictor\data\processed\aqi_features.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

DATASET LOADED
Shape: (17471, 82)
Columns: 82
Missing values: 0
Duplicate rows: 0


In [8]:
# ============================================================
# VERIFY MODEL FEATURES
# ============================================================

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing model features: {missing_features}"
    )

print("Model features found:", len(feature_columns))
print("Target found:", TARGET in df.columns)

print("\nFirst timestamp:", df["timestamp"].min())
print("Last timestamp:", df["timestamp"].max())

Model features found: 70
Target found: True

First timestamp: 2024-08-04 00:00:00+00:00
Last timestamp: 2026-08-01 22:00:00+00:00


In [9]:
# ============================================================
# STEP 4: CHRONOLOGICAL ORDER
# ============================================================

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Chronological order verified.")

print(
    "First:",
    df["timestamp"].iloc[0]
)

print(
    "Last:",
    df["timestamp"].iloc[-1]
)

Chronological order verified.
First: 2024-08-04 00:00:00+00:00
Last: 2026-08-01 22:00:00+00:00


In [10]:
# ============================================================
# STEP 5: X AND y
# ============================================================

X = df[feature_columns].copy()
y = df[TARGET].copy()

print("=" * 60)
print("MODEL DATA")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Number of features:", X.shape[1])

MODEL DATA
X shape: (17471, 70)
y shape: (17471,)
Number of features: 70


In [11]:
# ============================================================
# STEP 6: CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print("=" * 60)
print("CHRONOLOGICAL DATA SPLIT")
print("=" * 60)

print("Training:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTesting:")
print("X:", X_test.shape)
print("y:", y_test.shape)

print("\nDate ranges:")

print(
    "Train:",
    df["timestamp"].iloc[0],
    "→",
    df["timestamp"].iloc[train_end - 1]
)

print(
    "Validation:",
    df["timestamp"].iloc[train_end],
    "→",
    df["timestamp"].iloc[val_end - 1]
)

print(
    "Test:",
    df["timestamp"].iloc[val_end],
    "→",
    df["timestamp"].iloc[-1]
)

CHRONOLOGICAL DATA SPLIT
Training:
X: (12229, 70)
y: (12229,)

Validation:
X: (2621, 70)
y: (2621,)

Testing:
X: (2621, 70)
y: (2621,)

Date ranges:
Train: 2024-08-04 00:00:00+00:00 → 2025-12-26 12:00:00+00:00
Validation: 2025-12-26 13:00:00+00:00 → 2026-04-14 17:00:00+00:00
Test: 2026-04-14 18:00:00+00:00 → 2026-08-01 22:00:00+00:00


In [12]:
# ============================================================
# RIDGE REGRESSION — BASELINE MODEL
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 60)
print("RIDGE REGRESSION TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# 1. SCALE FEATURES
# ------------------------------------------------------------
# IMPORTANT:
# Fit scaler ONLY on training data to prevent data leakage.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Testing shape:", X_test_scaled.shape)


# ------------------------------------------------------------
# 2. CREATE RIDGE MODEL
# ------------------------------------------------------------

ridge_model = Ridge(
    alpha=1.0
)

# ------------------------------------------------------------
# 3. TRAIN
# ------------------------------------------------------------

print("\nTraining Ridge Regression...")

ridge_model.fit(
    X_train_scaled,
    y_train
)

print("Ridge training completed.")


# ------------------------------------------------------------
# 4. PREDICTIONS
# ------------------------------------------------------------

y_train_pred = ridge_model.predict(X_train_scaled)
y_val_pred = ridge_model.predict(X_val_scaled)
y_test_pred = ridge_model.predict(X_test_scaled)


# ------------------------------------------------------------
# 5. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(y_true, y_pred, dataset_name):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    print(f"\n{dataset_name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "Dataset": dataset_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# ------------------------------------------------------------
# 6. EVALUATE
# ------------------------------------------------------------

ridge_train_results = evaluate_model(
    y_train,
    y_train_pred,
    "Training"
)

ridge_val_results = evaluate_model(
    y_val,
    y_val_pred,
    "Validation"
)

ridge_test_results = evaluate_model(
    y_test,
    y_test_pred,
    "Testing"
)


# ------------------------------------------------------------
# 7. SAVE MODEL + SCALER
# ------------------------------------------------------------

joblib.dump(
    ridge_model,
    "ridge_aqi_model.pkl"
)

joblib.dump(
    scaler,
    "ridge_scaler.pkl"
)

print("\n" + "=" * 60)
print("RIDGE MODEL SAVED")
print("=" * 60)

print("Model : ridge_aqi_model.pkl")
print("Scaler: ridge_scaler.pkl")

RIDGE REGRESSION TRAINING
Training shape: (12229, 70)
Validation shape: (2621, 70)
Testing shape: (2621, 70)

Training Ridge Regression...
Ridge training completed.

Training
----------------------------------------
MAE  : 2.1560
RMSE : 3.5333
R²   : 0.9855

Validation
----------------------------------------
MAE  : 1.5488
RMSE : 2.4077
R²   : 0.9933

Testing
----------------------------------------
MAE  : 2.7992
RMSE : 4.3062
R²   : 0.9816

RIDGE MODEL SAVED
Model : ridge_aqi_model.pkl
Scaler: ridge_scaler.pkl


In [13]:
# ============================================================
# RANDOM FOREST REGRESSION TRAINING
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib
import os

print("=" * 60)
print("RANDOM FOREST REGRESSION TRAINING")
print("=" * 60)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")
rf_model.fit(X_train, y_train)

print("Random Forest training completed.")


# ------------------------------------------------------------
# EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(model, X, y, name):

    predictions = model.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# ------------------------------------------------------------
# EVALUATE
# ------------------------------------------------------------

rf_train_results = evaluate_model(
    rf_model,
    X_train,
    y_train,
    "Training"
)

rf_val_results = evaluate_model(
    rf_model,
    X_val,
    y_val,
    "Validation"
)

rf_test_results = evaluate_model(
    rf_model,
    X_test,
    y_test,
    "Testing"
)


# ------------------------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------------------------

feature_importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance": rf_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

print("\n" + "=" * 60)
print("TOP 20 RANDOM FOREST FEATURES")
print("=" * 60)

print(feature_importance.head(20).to_string(index=False))


# ------------------------------------------------------------
# SAVE MODEL
# ------------------------------------------------------------

os.makedirs("models", exist_ok=True)

rf_model_path = "models/random_forest_aqi.pkl"

joblib.dump(
    rf_model,
    rf_model_path
)

print("\nRandom Forest model saved to:")
print(rf_model_path)

RANDOM FOREST REGRESSION TRAINING
Training shape: (12229, 70)
Validation shape: (2621, 70)
Testing shape: (2621, 70)

Training Random Forest...
Random Forest training completed.

Training
----------------------------------------
MAE  : 0.4862
RMSE : 0.9484
R²   : 0.9990

Validation
----------------------------------------
MAE  : 1.1767
RMSE : 2.1727
R²   : 0.9946

Testing
----------------------------------------
MAE  : 2.6022
RMSE : 4.4868
R²   : 0.9800

TOP 20 RANDOM FOREST FEATURES
                 feature  importance
               aqi_lag_1    0.135623
             aqi_3h_mean    0.112815
          pm2_5_24h_mean    0.106084
             aqi_6h_mean    0.074716
               aqi_lag_3    0.066258
            aqi_12h_mean    0.056749
            aqi_24h_mean    0.042657
              aqi_lag_24    0.040437
             ozone_lag_3    0.039835
           aqi_change_3h    0.034567
           pm10_24h_mean    0.033469
           aqi_change_6h    0.033163
           aqi_change_1h    0.

In [14]:
# ============================================================
# XGBOOST REGRESSION TRAINING
# ============================================================

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib
import os

print("=" * 60)
print("XGBOOST REGRESSION TRAINING")
print("=" * 60)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost...")

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("XGBoost training completed.")


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

def evaluate_xgb(model, X, y, name):

    predictions = model.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


xgb_train_results = evaluate_xgb(
    xgb_model,
    X_train,
    y_train,
    "Training"
)

xgb_val_results = evaluate_xgb(
    xgb_model,
    X_val,
    y_val,
    "Validation"
)

xgb_test_results = evaluate_xgb(
    xgb_model,
    X_test,
    y_test,
    "Testing"
)


# ------------------------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------------------------

xgb_importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance": xgb_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

print("\n" + "=" * 60)
print("TOP 20 XGBOOST FEATURES")
print("=" * 60)

print(
    xgb_importance.head(20).to_string(index=False)
)


# ------------------------------------------------------------
# SAVE MODEL
# ------------------------------------------------------------

os.makedirs("models", exist_ok=True)

xgb_model_path = "models/xgboost_aqi.pkl"

joblib.dump(
    xgb_model,
    xgb_model_path
)

print("\nXGBoost model saved to:")
print(xgb_model_path)

XGBOOST REGRESSION TRAINING
Training shape: (12229, 70)
Validation shape: (2621, 70)
Testing shape: (2621, 70)

Training XGBoost...
XGBoost training completed.

Training
----------------------------------------
MAE  : 0.3366
RMSE : 0.4496
R²   : 0.9998

Validation
----------------------------------------
MAE  : 0.6242
RMSE : 1.2303
R²   : 0.9983

Testing
----------------------------------------
MAE  : 1.3132
RMSE : 2.4022
R²   : 0.9943

TOP 20 XGBOOST FEATURES
               feature  importance
             aqi_lag_1    0.532401
           aqi_3h_mean    0.170881
        pm2_5_24h_mean    0.148682
           ozone_lag_3    0.048590
         aqi_change_1h    0.047316
         aqi_change_3h    0.011449
           ozone_lag_1    0.006447
         aqi_change_6h    0.005183
           ozone_lag_6    0.004148
           aqi_6h_mean    0.003576
                 ozone    0.002321
          aqi_12h_mean    0.002037
            aqi_lag_24    0.001186
          ozone_lag_24    0.000979
nitrogen_d

In [16]:
# ============================================================
# BiLSTM AQI FORECASTING
# 24-HOUR LOOKBACK
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib
import os

print("=" * 60)
print("BiLSTM AQI FORECASTING")
print("=" * 60)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

LOOKBACK = 24
EPOCHS = 50
BATCH_SIZE = 64

print("Lookback:", LOOKBACK, "hours")
print("Features:", len(feature_columns))


# ============================================================
# 1. PREPARE COMPLETE CHRONOLOGICAL DATA
# ============================================================

model_df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

features_all = model_df[feature_columns].copy()
target_all = model_df["target_aqi"].copy()


# ============================================================
# 2. IDENTIFY SPLIT INDICES
# ============================================================

n = len(model_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

print("\nSplit indices:")
print("Train end:", train_end)
print("Validation end:", val_end)
print("Total:", n)


# ============================================================
# 3. SCALE FEATURES
# ============================================================

# IMPORTANT:
# Fit scaler ONLY on training period.

bilstm_scaler = StandardScaler()

features_train = bilstm_scaler.fit_transform(
    features_all.iloc[:train_end]
)

features_val = bilstm_scaler.transform(
    features_all.iloc[train_end:val_end]
)

features_test = bilstm_scaler.transform(
    features_all.iloc[val_end:]
)

print("\nScaled feature shapes:")
print("Train:", features_train.shape)
print("Validation:", features_val.shape)
print("Test:", features_test.shape)


# ============================================================
# 4. CREATE SEQUENCES
# ============================================================

def create_sequences(X, y, lookback):

    X_seq = []
    y_seq = []

    for i in range(lookback, len(X)):

        X_seq.append(
            X[i - lookback:i]
        )

        y_seq.append(
            y[i]
        )

    return np.array(X_seq), np.array(y_seq)


# ------------------------------------------------------------
# TRAIN SEQUENCES
# ------------------------------------------------------------

y_train_full = target_all.iloc[:train_end].values

X_train_seq, y_train_seq = create_sequences(
    features_train,
    y_train_full,
    LOOKBACK
)


# ------------------------------------------------------------
# VALIDATION SEQUENCES
# ------------------------------------------------------------

# Include the previous 24 hours from training so the
# first validation prediction has its complete history.

val_start = train_end - LOOKBACK

val_features_with_history = features_all.iloc[
    val_start:val_end
]

val_features_scaled = bilstm_scaler.transform(
    val_features_with_history
)

val_targets = target_all.iloc[
    val_start:val_end
].values

X_val_seq, y_val_seq = create_sequences(
    val_features_scaled,
    val_targets,
    LOOKBACK
)


# ------------------------------------------------------------
# TEST SEQUENCES
# ------------------------------------------------------------

# Include previous 24 hours from validation so the first
# test prediction has its complete historical context.

test_start = val_end - LOOKBACK

test_features_with_history = features_all.iloc[
    test_start:
]

test_features_scaled = bilstm_scaler.transform(
    test_features_with_history
)

test_targets = target_all.iloc[
    test_start:
].values

X_test_seq, y_test_seq = create_sequences(
    test_features_scaled,
    test_targets,
    LOOKBACK
)


# ============================================================
# 5. VERIFY SEQUENCES
# ============================================================

print("\n" + "=" * 60)
print("SEQUENCE SHAPES")
print("=" * 60)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)


# ============================================================
# 6. BUILD BiLSTM
# ============================================================

tf.random.set_seed(42)
np.random.seed(42)

bilstm_model = Sequential([
    
    Input(
        shape=(LOOKBACK, len(feature_columns))
    ),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(
            32,
            return_sequences=False
        )
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1
    )
])


# ============================================================
# 7. COMPILE
# ============================================================

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

print("\n" + "=" * 60)
print("BiLSTM ARCHITECTURE")
print("=" * 60)

bilstm_model.summary()


# ============================================================
# 8. CALLBACKS
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)


# ============================================================
# 9. TRAIN
# ============================================================

print("\n" + "=" * 60)
print("TRAINING BiLSTM")
print("=" * 60)

history = bilstm_model.fit(
    X_train_seq,
    y_train_seq,

    validation_data=(
        X_val_seq,
        y_val_seq
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


# ============================================================
# 10. EVALUATION
# ============================================================

def evaluate_bilstm(model, X, y, name):

    predictions = model.predict(
        X,
        verbose=0
    ).flatten()

    mae = mean_absolute_error(
        y,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y,
            predictions
        )
    )

    r2 = r2_score(
        y,
        predictions
    )

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "predictions": predictions
    }


# ============================================================
# 11. TRAIN / VALIDATION / TEST RESULTS
# ============================================================

bilstm_train_results = evaluate_bilstm(
    bilstm_model,
    X_train_seq,
    y_train_seq,
    "Training"
)

bilstm_val_results = evaluate_bilstm(
    bilstm_model,
    X_val_seq,
    y_val_seq,
    "Validation"
)

bilstm_test_results = evaluate_bilstm(
    bilstm_model,
    X_test_seq,
    y_test_seq,
    "Testing"
)


# ============================================================
# 12. SAVE MODEL AND SCALER
# ============================================================

os.makedirs(
    "models",
    exist_ok=True
)

bilstm_model.save(
    "models/bilstm_aqi.keras"
)

joblib.dump(
    bilstm_scaler,
    "models/bilstm_scaler.pkl"
)

print("\n" + "=" * 60)
print("BiLSTM MODEL SAVED")
print("=" * 60)

print("Model : models/bilstm_aqi.keras")
print("Scaler: models/bilstm_scaler.pkl")

BiLSTM AQI FORECASTING
Lookback: 24 hours
Features: 70

Split indices:
Train end: 12229
Validation end: 14850
Total: 17471

Scaled feature shapes:
Train: (12229, 70)
Validation: (2621, 70)
Test: (2621, 70)

SEQUENCE SHAPES
X_train: (12205, 24, 70)
y_train: (12205,)
X_val: (2621, 24, 70)
y_val: (2621,)
X_test: (2621, 24, 70)
y_test: (2621,)

BiLSTM ARCHITECTURE


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 24, 128)        │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,449 (439.25 KB)

 Trainable params: 112,449 (439.25 KB)

 Non-trainable params: 0 (0.00 B)


TRAINING BiLSTM
Epoch 1/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - loss: 6107.3252 - mae: 66.9750 - val_loss: 1166.9310 - val_mae: 26.8262 - learning_rate: 0.0010
Epoch 2/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 12s 60ms/step - loss: 836.7693 - mae: 23.0319 - val_loss: 333.6376 - val_mae: 12.4546 - learning_rate: 0.0010
Epoch 3/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - loss: 200.0935 - mae: 10.0714 - val_loss: 44.1871 - val_mae: 4.6637 - learning_rate: 0.0010
Epoch 4/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - loss: 108.8382 - mae: 7.7827 - val_loss: 21.0628 - val_mae: 3.3943 - learning_rate: 0.0010
Epoch 5/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - loss: 95.1414 - mae: 7.4331 - val_loss: 14.6196 - val_mae: 2.7611 - learning_rate: 0.0010
Epoch 6/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 53ms/step - loss: 84.9433 - mae: 7.1077 - val_loss: 11.9716 - val_mae: 2.4476 - learning_rate: 0.0010
Epoch 7/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - loss: 80.6256 - mae: 6.9189 - val_loss:

In [17]:
# ============================================================
# TARGET LEAKAGE VERIFICATION
# ============================================================

print("=" * 60)
print("TARGET LEAKAGE VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check whether target itself is included
# ------------------------------------------------------------

print("\n1. TARGET COLUMN CHECK")

if "target_aqi" in feature_columns:
    print(" LEAKAGE: target_aqi is included in feature_columns")
else:
    print(" target_aqi is NOT included in feature_columns")


# ------------------------------------------------------------
# 2. Check suspicious future/current target-related columns
# ------------------------------------------------------------

print("\n2. TARGET-RELATED FEATURE CHECK")

target_related = [
    col for col in feature_columns
    if "target" in col.lower()
]

if target_related:
    print(" Target-related columns found:")
    for col in target_related:
        print("  ", col)
else:
    print("No target-related feature names found")


# ------------------------------------------------------------
# 3. Check whether feature values are shifted correctly
# ------------------------------------------------------------

print("\n3. LAG FEATURE CHECK")

lag_features = [
    col for col in feature_columns
    if "_lag_" in col
]

print("Number of lag features:", len(lag_features))

for col in lag_features:
    print("  ✓", col)


# ------------------------------------------------------------
# 4. Check rolling features
# ------------------------------------------------------------

print("\n4. ROLLING FEATURE CHECK")

rolling_features = [
    col for col in feature_columns
    if "_mean" in col
]

print("Number of rolling mean features:", len(rolling_features))

for col in rolling_features:
    print("  ✓", col)


# ------------------------------------------------------------
# 5. Check change features
# ------------------------------------------------------------

print("\n5. CHANGE FEATURE CHECK")

change_features = [
    col for col in feature_columns
    if "_change_" in col
]

print("Number of change features:", len(change_features))

for col in change_features:
    print("  ✓", col)


# ------------------------------------------------------------
# 6. Check timestamp ordering
# ------------------------------------------------------------

print("\n6. CHRONOLOGICAL ORDER CHECK")

timestamps = pd.to_datetime(model_df["timestamp"])

if timestamps.is_monotonic_increasing:
    print(" Data is chronologically ordered")
else:
    print(" Data is NOT chronologically ordered")


# ------------------------------------------------------------
# 7. Check feature count
# ------------------------------------------------------------

print("\n7. FEATURE COUNT")

print("Feature columns:", len(feature_columns))

if len(feature_columns) == 70:
    print(" Exactly 70 model features")
else:
    print(" Expected 70 features, found:", len(feature_columns))


# ------------------------------------------------------------
# 8. Check target statistics
# ------------------------------------------------------------

print("\n8. TARGET INFORMATION")

print("Target:", "target_aqi")
print("Target minimum:", model_df["target_aqi"].min())
print("Target maximum:", model_df["target_aqi"].max())
print("Target mean:", model_df["target_aqi"].mean())


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LEAKAGE CHECK SUMMARY")
print("=" * 60)

print("Target included in X:", "target_aqi" in feature_columns)
print("Number of model features:", len(feature_columns))
print("Chronological order:", timestamps.is_monotonic_increasing)

print("\n IMPORTANT:")
print("This confirms structural leakage checks.")
print("The feature-generation code should also be inspected")
print("to confirm that rolling/lag features use only past data.")

TARGET LEAKAGE VERIFICATION

1. TARGET COLUMN CHECK
 target_aqi is NOT included in feature_columns

2. TARGET-RELATED FEATURE CHECK
No target-related feature names found

3. LAG FEATURE CHECK
Number of lag features: 31
  ✓ aqi_lag_1
  ✓ aqi_lag_3
  ✓ aqi_lag_6
  ✓ aqi_lag_12
  ✓ aqi_lag_24
  ✓ aqi_lag_48
  ✓ aqi_lag_72
  ✓ pm2_5_lag_1
  ✓ pm2_5_lag_3
  ✓ pm2_5_lag_6
  ✓ pm2_5_lag_24
  ✓ pm10_lag_1
  ✓ pm10_lag_3
  ✓ pm10_lag_6
  ✓ pm10_lag_24
  ✓ carbon_monoxide_lag_1
  ✓ carbon_monoxide_lag_3
  ✓ carbon_monoxide_lag_6
  ✓ carbon_monoxide_lag_24
  ✓ nitrogen_dioxide_lag_1
  ✓ nitrogen_dioxide_lag_3
  ✓ nitrogen_dioxide_lag_6
  ✓ nitrogen_dioxide_lag_24
  ✓ sulphur_dioxide_lag_1
  ✓ sulphur_dioxide_lag_3
  ✓ sulphur_dioxide_lag_6
  ✓ sulphur_dioxide_lag_24
  ✓ ozone_lag_1
  ✓ ozone_lag_3
  ✓ ozone_lag_6
  ✓ ozone_lag_24

4. ROLLING FEATURE CHECK
Number of rolling mean features: 14
  ✓ aqi_3h_mean
  ✓ aqi_6h_mean
  ✓ aqi_12h_mean
  ✓ aqi_24h_mean
  ✓ pm2_5_3h_mean
  ✓ pm2_5_6h_mean
  ✓ p

In [22]:
# ============================================================
# FINAL TEMPORAL LEAKAGE SANITY CHECK
# ============================================================

print("=" * 60)
print("FINAL TEMPORAL LEAKAGE SANITY CHECK")
print("=" * 60)

# Sort chronologically
df_check = df.sort_values("timestamp").reset_index(drop=True)

# ------------------------------------------------------------
# 1. TARGET MUST BE EXACTLY THE NEXT-HOUR AQI
# ------------------------------------------------------------

expected_target = df_check["us_aqi"].shift(-1)

target_matches = (
    df_check["target_aqi"].iloc[:-1].reset_index(drop=True)
    ==
    expected_target.iloc[:-1].reset_index(drop=True)
).all()

print("\n1. NEXT-HOUR TARGET CHECK")
print("Target equals next-hour us_aqi:", target_matches)

# ------------------------------------------------------------
# 2. CHECK AQI LAGS
# ------------------------------------------------------------

lag_checks = {
    "aqi_lag_1": 1,
    "aqi_lag_3": 3,
    "aqi_lag_6": 6,
    "aqi_lag_12": 12,
    "aqi_lag_24": 24,
    "aqi_lag_48": 48,
    "aqi_lag_72": 72,
}

print("\n2. AQI LAG CHECKS")

for feature, lag in lag_checks.items():

    expected = df_check["us_aqi"].shift(lag)

    valid = (
        df_check[feature].notna()
        & expected.notna()
    )

    passed = (
        df_check.loc[valid, feature].values
        == expected.loc[valid].values
    ).all()

    print(
        f"{feature:<15} -> "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        all_passed = False

# ------------------------------------------------------------
# 3. CHECK THAT TARGET IS NOT IN FEATURES
# ------------------------------------------------------------

print("\n3. TARGET IN FEATURE CHECK")

print(
    "target_aqi in X:",
    "target_aqi" in feature_columns
)

print(
    "us_aqi in X:",
    "us_aqi" in feature_columns
)

# ------------------------------------------------------------
# 4. CHECK CHRONOLOGICAL ORDER
# ------------------------------------------------------------

print("\n4. CHRONOLOGICAL ORDER CHECK")

is_sorted = df_check["timestamp"].is_monotonic_increasing

print(
    "Chronologically ordered:",
    is_sorted
)

# ------------------------------------------------------------
# 5. CHECK FUTURE INFORMATION
# ------------------------------------------------------------

print("\n5. FUTURE INFORMATION CHECK")

future_keywords = [
    "future",
    "next",
    "target"
]

suspicious = [
    col for col in feature_columns
    if any(keyword in col.lower() for keyword in future_keywords)
]

if suspicious:
    print("Potentially suspicious features:")
    print(suspicious)
else:
    print("No future/target-named features found.")

# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

all_checks = (
    target_matches
    and is_sorted
    and "target_aqi" not in feature_columns
    and "us_aqi" not in feature_columns
    and not suspicious
)

if all_checks:
    print("✓ TEMPORAL LEAKAGE SANITY CHECK PASSED")
    print("✓ Target represents next-hour AQI")
    print("✓ Target is excluded from model features")
    print("✓ Current AQI is excluded from model features")
    print("✓ Data is chronologically ordered")
    print("✓ No suspicious future-named features detected")
else:
    print("⚠ REVIEW REQUIRED")

FINAL TEMPORAL LEAKAGE SANITY CHECK

1. NEXT-HOUR TARGET CHECK
Target equals next-hour us_aqi: True

2. AQI LAG CHECKS
aqi_lag_1       -> PASS
aqi_lag_3       -> PASS
aqi_lag_6       -> PASS
aqi_lag_12      -> PASS
aqi_lag_24      -> PASS
aqi_lag_48      -> PASS
aqi_lag_72      -> PASS

3. TARGET IN FEATURE CHECK
target_aqi in X: False
us_aqi in X: False

4. CHRONOLOGICAL ORDER CHECK
Chronologically ordered: True

5. FUTURE INFORMATION CHECK
No future/target-named features found.

FINAL RESULT
✓ TEMPORAL LEAKAGE SANITY CHECK PASSED
✓ Target represents next-hour AQI
✓ Target is excluded from model features
✓ Current AQI is excluded from model features
✓ Data is chronologically ordered
✓ No suspicious future-named features detected
